In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb

# ==========================================
# データの読み込みと初期処理
# ==========================================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# 1. 訓練データの欠損値の割合を計算
missing_ratio = train.isnull().sum() / len(train)

# 2. 欠損率が 80% を超えている列の名前を抽出
cols_to_drop = missing_ratio[missing_ratio > 0.8].index.tolist()
print(f"🗑️ 欠損値が多すぎるため削除する列: {cols_to_drop}")
# 出力例: ['Alley', 'PoolQC', 'Fence', 'MiscFeature']

# 3. これらの列を train_df, test_df からドロップする
train = train.drop(columns=cols_to_drop, errors='ignore')
test = test.drop(columns=cols_to_drop, errors='ignore')

# 正解ラベルの退避と、IDの削除
y = train["SalePrice"]
train_df = train.drop(columns=["Id", "SalePrice"])
test_df = test.drop(columns=["Id"])

# 後の分割のために訓練データの行数を覚えておく
num_train = len(train_df)

# 【ポイント】前処理のために一度 train と test を合体させる！
all_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

# ==========================================
# 一発で終わる前処理
# ==========================================
# 文字列の列（object型）を自動抽出して、一括で category型 に変換
cat_cols = all_df.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:
    all_df[col] = all_df[col].astype("category")

# 合体させていたデータを、元の訓練データとテストデータに綺麗に切り戻す
X = all_df.iloc[:num_train].copy()
X_test = all_df.iloc[num_train:].copy()

# ==========================================
# 5-Fold 交差検証 & 学習ループ
# ==========================================


# ==========================================
# 1. 予測結果を貯める器をモデルごとに用意
# ==========================================
# 【モデルA: LightGBM】用
oof_lgb = np.zeros(num_train)
test_lgb = np.zeros(len(X_test))

# 【モデルB: Ridge回帰（線形モデル）】用
oof_ridge = np.zeros(num_train)
test_ridge = np.zeros(len(X_test))

# 【モデルC: ランダムフォレスト（決定木）】用
oof_rf = np.zeros(num_train)
test_rf = np.zeros(len(X_test))

# ==========================================
# 2. 5-Fold 交差検証のループ
# ==========================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"--- Fold {fold + 1} / 5 ---")
    
    # データの分割（前回のバグ対策をすべて盛り込んだ綺麗な状態）
    X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx].copy(), y.iloc[val_idx]
    
    # y を確実な1次元配列にする
    y_train_series = y_train.iloc[:, 0] if isinstance(y_train, pd.DataFrame) else y_train
    y_val_series = y_val.iloc[:, 0] if isinstance(y_val, pd.DataFrame) else y_val
    
    # X_testの型を安全にDataFrame化
    X_test_fold = pd.DataFrame(X_test.copy())
    if isinstance(X_test, pd.Series):
        X_test_fold.columns = ['Neighborhood']
        
    # --- 💡 ターゲットエンコーディング（TE）処理 ---
    target_mean = y_train_series.groupby(X_train['Neighborhood']).mean()
    global_mean = float(y_train_series.mean())
    
    s_train = pd.Series(X_train['Neighborhood'])
    s_val = pd.Series(X_val['Neighborhood'])
    s_test = pd.Series(X_test_fold['Neighborhood'])
    
    X_train["TE_Neighborhood"] = s_train.map(target_mean).astype(float).fillna(global_mean)
    X_val["TE_Neighborhood"] = s_val.map(target_mean).astype(float).fillna(global_mean)
    X_test_fold["TE_Neighborhood"] = s_test.map(target_mean).astype(float).fillna(global_mean)
    
    # --- 💡 scikit-learn用の前処理（文字列列を削除・またはダミー化） ---
    # RidgeやRandomForestは文字列（category型）を読めないので、TE化した列や数値列だけを抽出します
    num_cols = X_train.select_dtypes(include=[np.number]).columns
    X_train_sk = X_train[num_cols].fillna(0)
    X_val_sk = X_val[num_cols].fillna(0)
    X_test_sk = X_test_fold[num_cols].fillna(0)

    # ------------------------------------------
    # モデルA: LightGBM の学習・予測
    # ------------------------------------------
    model_lgb = lgb.LGBMRegressor(random_state=42, n_estimators=500, learning_rate=0.05, verbose=-1)
    actual_cat_cols = [c for c in cat_cols if c in X_train.columns]
    model_lgb.fit(X_train, y_train, categorical_feature=actual_cat_cols, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
    
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    test_lgb += model_lgb.predict(X_test_fold) / 5

    # ------------------------------------------
    # モデルB: Ridge回帰（scikit-learn）の学習・予測
    # ------------------------------------------
    model_ridge = Ridge(alpha=1.0, random_state=42)
    model_ridge.fit(X_train_sk, y_train_series)
    
    oof_ridge[val_idx] = model_ridge.predict(X_val_sk)
    test_ridge += model_ridge.predict(X_test_sk) / 5

    # ------------------------------------------
    # モデルC: ランダムフォレスト（scikit-learn）の学習・予測
    # ------------------------------------------
    model_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model_rf.fit(X_train_sk, y_train_series)
    
    oof_rf[val_idx] = model_rf.predict(X_val_sk)
    test_rf += model_rf.predict(X_test_sk) / 5

print("🎉 すべてのモデルの5折学習が完了しました！")

# ==========================================
# 3. 複数のモデルを組み合わせる（ブレンド）
# ==========================================
# それぞれのモデルの予測値を「黄金比率」で混ぜ合わせます（合計で1.0になるようにします）
final_oof_preds = (oof_lgb * 0.6) + (oof_ridge * 0.1) + (oof_rf * 0.3)
final_test_preds = (test_lgb * 0.6) + (test_ridge * 0.1) + (test_rf * 0.3)

# ==========================================
# 4. 最終的な総合スコア（RMSEなど）を計算
# ==========================================
from sklearn.metrics import root_mean_squared_error
# ※お使いの環境のsklearnが古い場合は、以下を使用してください：
# from sklearn.metrics import mean_squared_error
# rmse = np.sqrt(mean_squared_error(y, final_oof_preds))

rmse = root_mean_squared_error(y, final_oof_preds)
print(f"📊 3モデル融合後の総合RMSEスコア: {rmse:.4f}")

🗑️ 欠損値が多すぎるため削除する列: ['Alley', 'PoolQC', 'Fence', 'MiscFeature']
--- Fold 1 / 5 ---


C:\Users\user\AppData\Local\Temp\ipykernel_18664\4042436234.py:91: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_mean = y_train_series.groupby(X_train['Neighborhood']).mean()


--- Fold 2 / 5 ---


C:\Users\user\AppData\Local\Temp\ipykernel_18664\4042436234.py:91: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_mean = y_train_series.groupby(X_train['Neighborhood']).mean()


--- Fold 3 / 5 ---


...

--- Fold 4 / 5 ---


C:\Users\user\AppData\Local\Temp\ipykernel_18664\4042436234.py:91: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_mean = y_train_series.groupby(X_train['Neighborhood']).mean()


--- Fold 5 / 5 ---


C:\Users\user\AppData\Local\Temp\ipykernel_18664\4042436234.py:91: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_mean = y_train_series.groupby(X_train['Neighborhood']).mean()


🎉 すべてのモデルの5折学習が完了しました！
📊 3モデル融合後の総合RMSEスコア: 28553.8808
